# Legitimate vs Malware Type Classification

This notebook creates a dataset with Legitimate | Malware Type structure and trains classification models.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load dataset from Final_cap directory
dataset_path = r"C:\Users\User\Desktop\Final_cap\dataset\merged_dataset.csv"

if os.path.exists(dataset_path):
    df = pd.read_csv(dataset_path, sep='|')
    print(f"Dataset shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
else:
    print(f"Dataset not found at: {dataset_path}")
    print("Please check the path")

In [ ]:
# Create Legitimate | Malware Type structure
def create_combined_labels(df):
    """Create combined labels: Legitimate or specific malware types"""
    if 'legitimate' in df.columns and 'MalwareType' in df.columns:
        df['Combined_Label'] = df.apply(lambda row: 
            'Legitimate' if row['legitimate'] == 0 
            else f"{row['MalwareType']}", axis=1)
    else:
        print("Required columns 'legitimate' and 'MalwareType' not found")
        return df
    
    return df

df = create_combined_labels(df)

if 'Combined_Label' in df.columns:
    print("\nCombined Label Distribution:")
    print(df['Combined_Label'].value_counts())
    
    # Visualize distribution
    plt.figure(figsize=(12, 6))
    df['Combined_Label'].value_counts().plot(kind='bar')
    plt.title('Distribution of Legitimate vs Malware Types')
    plt.xlabel('Label')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Prepare features and target
# Remove non-feature columns
exclude_cols = ['Name', 'md5', 'legitimate', 'MalwareType', 'Combined_Label']
feature_columns = [col for col in df.columns if col not in exclude_cols]

X = df[feature_columns]
y = df['Combined_Label']

print(f"Features shape: {X.shape}")
print(f"Number of classes: {len(y.unique())}")
print(f"Classes: {y.unique()}")

In [ ]:
# Handle missing values and preprocessing
print(f"Missing values: {X.isnull().sum().sum()}")

# Fill missing values with median
X = X.fillna(X.median())

# Encode target labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"Encoded classes: {label_encoder.classes_}")
print(f"Class distribution:")
unique, counts = np.unique(y_encoded, return_counts=True)
for i, (cls, count) in enumerate(zip(label_encoder.classes_, counts)):
    print(f"  {cls}: {count}")

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Train Extra Trees Classifier
print("Training Extra Trees Classifier...")
extra_trees = ExtraTreesClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'  # Handle class imbalance
)
extra_trees.fit(X_train, y_train)

et_pred = extra_trees.predict(X_test)
et_accuracy = accuracy_score(y_test, et_pred)

print(f"Extra Trees Accuracy: {et_accuracy:.4f}")

In [ ]:
# Train Logistic Regression
print("Training Logistic Regression...")
logistic_reg = LogisticRegression(
    random_state=42,
    max_iter=1000,
    multi_class='ovr',
    class_weight='balanced'  # Handle class imbalance
)
logistic_reg.fit(X_train_scaled, y_train)

lr_pred = logistic_reg.predict(X_test_scaled)
lr_accuracy = accuracy_score(y_test, lr_pred)

print(f"Logistic Regression Accuracy: {lr_accuracy:.4f}")

In [ ]:
# Detailed evaluation
print("=== Extra Trees Results ===")
print(classification_report(y_test, et_pred, target_names=label_encoder.classes_))

print("\n=== Logistic Regression Results ===")
print(classification_report(y_test, lr_pred, target_names=label_encoder.classes_))

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Extra Trees confusion matrix
cm_et = confusion_matrix(y_test, et_pred)
sns.heatmap(cm_et, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_encoder.classes_, 
            yticklabels=label_encoder.classes_, ax=axes[0])
axes[0].set_title('Extra Trees - Confusion Matrix')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Logistic Regression confusion matrix
cm_lr = confusion_matrix(y_test, lr_pred)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Greens', 
            xticklabels=label_encoder.classes_, 
            yticklabels=label_encoder.classes_, ax=axes[1])
axes[1].set_title('Logistic Regression - Confusion Matrix')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

In [ ]:
# Save the restructured dataset and models
import joblib

# Create output directory
os.makedirs('models', exist_ok=True)
os.makedirs('dataset', exist_ok=True)

# Save restructured dataset
df.to_csv('dataset/legitimate_malware_dataset.csv', sep='|', index=False)

# Save models
joblib.dump(extra_trees, 'models/extra_trees_legitimate_malware.pkl')
joblib.dump(logistic_reg, 'models/logistic_regression_legitimate_malware.pkl')
joblib.dump(scaler, 'models/scaler_legitimate_malware.pkl')
joblib.dump(label_encoder, 'models/label_encoder_legitimate_malware.pkl')

print("Dataset and models saved successfully!")
print(f"Extra Trees Accuracy: {et_accuracy:.4f}")
print(f"Logistic Regression Accuracy: {lr_accuracy:.4f}")